# Cat, Talk To Me - cat-face detector training

This notebook fine-tunes **YOLOX-Nano** (Megvii, Apache-2.0) to find a **cat's face** in a photo, robust to tilted heads, partial occlusion, and fur that looks like a face. It replaces the OpenCV Haar cascades in `server/app/detect.py`, which miss tilted faces and fire on fur.

**Runtime:** Colab, GPU (T4 is enough). Menu: *Runtime → Change runtime type → T4 GPU*. Training takes roughly 45-70 minutes.

**What you do:** run the cells top to bottom. One cell asks for your **Roboflow API key** (from https://app.roboflow.com/settings/api) with a hidden prompt. It is used once to download the dataset and is not saved in the notebook. At the end, the notebook downloads three files to your computer: the ONNX model, a model card, and the evaluation numbers. Put the `.onnx` file in `server/models/`.

## Licenses (all permissive; attribution required)
| Thing | License | Attribution |
|---|---|---|
| Dataset: Roboflow Universe "cat-face-data" (workspace `cat-face`), 8,153 images | CC BY 4.0 | footer + README credit |
| YOLOX code and COCO-pretrained `yolox_nano.pth` | Apache-2.0 | README credit |
| The fine-tuned model this notebook produces | yours; derived from the above | keep both credits |

The notebook prints the dataset's license as reported by the Roboflow API so you can confirm it before training.


In [ ]:
#@title 1. Check the GPU
import subprocess, sys
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout or "No GPU! Runtime -> Change runtime type -> T4 GPU")


In [ ]:
#@title 2. Install YOLOX, Roboflow client, ONNX tools (2-4 min)
%cd /content
!git clone -q --depth 1 --branch 0.3.0 https://github.com/Megvii-BaseDetection/YOLOX.git
%cd /content/YOLOX
!pip install -q -e . 2>&1 | tail -2
!pip install -q roboflow onnx onnxsim opencv-python-headless pycocotools 2>&1 | tail -1
import yolox, cv2, torch
print("yolox", yolox.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| opencv", cv2.__version__)


In [ ]:
#@title 3. Download the dataset (COCO format) - asks for your Roboflow API key
from getpass import getpass
from roboflow import Roboflow
import os, json, glob

api_key = getpass("Roboflow API key (hidden, not saved): ")
rf = Roboflow(api_key=api_key)
del api_key
project = rf.workspace("cat-face").project("cat-face-data")
versions = project.versions()
version = versions[-1]
print("Project:", project.name, "| versions:", [v.version for v in versions], "| using version", version.version)
meta = project.__dict__
for k in ("license", "public", "type", "annotation", "images", "classes"):
    if k in meta: print(f"  {k}: {meta[k]}")
ds = version.download("coco", location="/content/catface_raw", overwrite=True)
print("downloaded to", ds.location)
for split in ("train", "valid", "test"):
    j = f"{ds.location}/{split}/_annotations.coco.json"
    if os.path.exists(j):
        d = json.load(open(j))
        print(f"  {split}: {len(d['images'])} images, {len(d['annotations'])} boxes, categories={[c['name'] for c in d['categories']]}")


### Confirm before you continue
The cell above must print a license of **CC BY 4.0** (or the Universe page must show it: https://universe.roboflow.com/cat-face/cat-face-data). If it prints something else, stop and tell Claude.

In [ ]:
#@title 4. Normalise categories to one class "cat_face" and lay the data out for YOLOX
import shutil, json, os
RAW = "/content/catface_raw"
DATA = "/content/YOLOX/datasets/catface"
os.makedirs(f"{DATA}/annotations", exist_ok=True)

def normalise(split_in, split_out):
    src = f"{RAW}/{split_in}"
    d = json.load(open(f"{src}/_annotations.coco.json"))
    # Roboflow exports often include a parent "superclass" category with no boxes. Collapse everything to id 1.
    cats_with_boxes = {a["category_id"] for a in d["annotations"]}
    print(f"{split_in}: categories in file={[(c['id'], c['name']) for c in d['categories']]} used by boxes={sorted(cats_with_boxes)}")
    for a in d["annotations"]:
        a["category_id"] = 1
        a["iscrowd"] = a.get("iscrowd", 0)
    d["categories"] = [{"id": 1, "name": "cat_face", "supercategory": "cat"}]
    out_dir = f"{DATA}/{split_out}"
    if os.path.exists(out_dir): shutil.rmtree(out_dir)
    shutil.copytree(src, out_dir, ignore=shutil.ignore_patterns("*.json"))
    json.dump(d, open(f"{DATA}/annotations/{split_out}.json", "w"))
    return d

train = normalise("train", "train")
val = normalise("valid", "val")
if os.path.exists(f"{RAW}/test"): test = normalise("test", "test")
# box-size statistics tell us what minSize the server can expect
import numpy as np
ws = np.array([a["bbox"][2] for a in train["annotations"]]); hs = np.array([a["bbox"][3] for a in train["annotations"]])
print(f"train box width px: min {ws.min():.0f}  p5 {np.percentile(ws,5):.0f}  median {np.median(ws):.0f}  p95 {np.percentile(ws,95):.0f}")
print(f"aspect w/h: median {np.median(ws/hs):.2f}")


In [ ]:
#@title 5. Rotation augmentation: rotated copies of 60% of the training images, boxes corrected
import cv2, json, math, random, os, numpy as np
random.seed(7)
DATA = "/content/YOLOX/datasets/catface"
d = json.load(open(f"{DATA}/annotations/train.json"))
by_img = {}
for a in d["annotations"]: by_img.setdefault(a["image_id"], []).append(a)
next_img_id = max(i["id"] for i in d["images"]) + 1
next_ann_id = max(a["id"] for a in d["annotations"]) + 1
new_imgs, new_anns = [], []

def rotate_with_boxes(img, boxes_xywh, deg):
    h, w = img.shape[:2]
    m = cv2.getRotationMatrix2D((w/2, h/2), deg, 1.0)
    # expand the canvas so nothing is cropped
    cos, sin = abs(m[0,0]), abs(m[0,1])
    nw, nh = int(h*sin + w*cos), int(h*cos + w*sin)
    m[0,2] += nw/2 - w/2; m[1,2] += nh/2 - h/2
    out = cv2.warpAffine(img, m, (nw, nh), borderMode=cv2.BORDER_REPLICATE)
    new_boxes = []
    for x, y, bw, bh in boxes_xywh:
        pts = np.array([[x,y],[x+bw,y],[x,y+bh],[x+bw,y+bh]], dtype=np.float32)
        pts = np.hstack([pts, np.ones((4,1), np.float32)]) @ m.T
        x0, y0 = pts.min(0); x1, y1 = pts.max(0)
        x0, y0 = max(0, x0), max(0, y0); x1, y1 = min(nw, x1), min(nh, y1)
        new_boxes.append([float(x0), float(y0), float(x1-x0), float(y1-y0)])
    return out, new_boxes

chosen = [im for im in d["images"] if random.random() < 0.6]
for im in chosen:
    path = f"{DATA}/train/{im['file_name']}"
    img = cv2.imread(path)
    if img is None: continue
    deg = random.choice([-1, 1]) * random.uniform(15, 45)
    anns = by_img.get(im["id"], [])
    out, boxes = rotate_with_boxes(img, [a["bbox"] for a in anns], deg)
    fname = f"rot{deg:+.0f}_{im['file_name']}".replace(" ", "_")
    cv2.imwrite(f"{DATA}/train/{fname}", out, [cv2.IMWRITE_JPEG_QUALITY, 92])
    new_imgs.append({"id": next_img_id, "file_name": fname, "width": out.shape[1], "height": out.shape[0]})
    for a, b in zip(anns, boxes):
        if b[2] < 4 or b[3] < 4: continue
        new_anns.append({"id": next_ann_id, "image_id": next_img_id, "category_id": 1, "bbox": b, "area": b[2]*b[3], "iscrowd": 0})
        next_ann_id += 1
    next_img_id += 1
d["images"] += new_imgs; d["annotations"] += new_anns
json.dump(d, open(f"{DATA}/annotations/train.json", "w"))
print(f"added {len(new_imgs)} rotated images / {len(new_anns)} boxes -> train now {len(d['images'])} images, {len(d['annotations'])} boxes")

# a rotated copy of the validation split, to measure rotation robustness honestly
v = json.load(open(f"{DATA}/annotations/val.json"))
os.makedirs(f"{DATA}/val_rot", exist_ok=True)
vb = {}
for a in v["annotations"]: vb.setdefault(a["image_id"], []).append(a)
rot_anns = []; aid = 1
for im in v["images"]:
    img = cv2.imread(f"{DATA}/val/{im['file_name']}")
    if img is None: continue
    deg = random.choice([-1, 1]) * random.uniform(20, 40)
    anns = vb.get(im["id"], [])
    out, boxes = rotate_with_boxes(img, [a["bbox"] for a in anns], deg)
    cv2.imwrite(f"{DATA}/val_rot/{im['file_name']}", out, [cv2.IMWRITE_JPEG_QUALITY, 92])
    im["width"], im["height"] = out.shape[1], out.shape[0]
    for b in boxes:
        rot_anns.append({"id": aid, "image_id": im["id"], "category_id": 1, "bbox": b, "area": b[2]*b[3], "iscrowd": 0}); aid += 1
v["annotations"] = rot_anns
json.dump(v, open(f"{DATA}/annotations/val_rot.json", "w"))
print(f"rotated validation copy: {len(v['images'])} images")

# show three augmented samples with their boxes
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, im in zip(axes, random.sample(new_imgs, 3)):
    img = cv2.cvtColor(cv2.imread(f"{DATA}/train/{im['file_name']}"), cv2.COLOR_BGR2RGB)
    for a in new_anns:
        if a["image_id"] == im["id"]:
            x, y, w, h = map(int, a["bbox"]); cv2.rectangle(img, (x, y), (x+w, y+h), (255, 0, 0), 3)
    ax.imshow(img); ax.set_title(im["file_name"][:20]); ax.axis("off")
plt.show()


In [ ]:
#@title 6. Write the YOLOX experiment file
exp_src = '''
import os
from yolox.exp import Exp as BaseExp

class Exp(BaseExp):
    def __init__(self):
        super().__init__()
        # ---- YOLOX-Nano architecture ----
        self.depth = 0.33
        self.width = 0.25
        self.depthwise = True
        self.input_size = (416, 416)
        self.test_size = (416, 416)
        self.random_size = (10, 20)
        self.mosaic_scale = (0.5, 1.5)
        self.enable_mixup = False
        # ---- our data ----
        self.num_classes = 1
        self.data_dir = "/content/YOLOX/datasets/catface"
        self.train_ann = "train.json"
        self.val_ann = "val.json"
        self.train_name = "train"
        self.val_name = "val"
        # ---- schedule ----
        self.max_epoch = 30
        self.no_aug_epochs = 5
        self.warmup_epochs = 1
        self.eval_interval = 5
        self.data_num_workers = 4
        self.flip_prob = 0.5
        self.hsv_prob = 1.0
        self.degrees = 0.0     # rotation was done offline with corrected boxes
        self.exp_name = "catface_nano"

    def get_model(self, sublinear=False):
        import torch.nn as nn
        from yolox.models import YOLOX, YOLOPAFPN, YOLOXHead
        def init_yolo(M):
            for m in M.modules():
                if isinstance(m, nn.BatchNorm2d):
                    m.eps = 1e-3
                    m.momentum = 0.03
        if getattr(self, "model", None) is None:
            in_channels = [256, 512, 1024]
            backbone = YOLOPAFPN(self.depth, self.width, in_channels=in_channels, act=self.act, depthwise=True)
            head = YOLOXHead(self.num_classes, self.width, in_channels=in_channels, act=self.act, depthwise=True)
            self.model = YOLOX(backbone, head)
        self.model.apply(init_yolo)
        self.model.head.initialize_biases(1e-2)
        self.model.train()
        return self.model

    def get_dataset(self, cache=False, cache_type="ram"):
        from yolox.data import COCODataset, TrainTransform
        return COCODataset(data_dir=self.data_dir, json_file=self.train_ann, name=self.train_name,
                           img_size=self.input_size,
                           preproc=TrainTransform(max_labels=50, flip_prob=self.flip_prob, hsv_prob=self.hsv_prob),
                           cache=cache, cache_type=cache_type)

    def get_eval_dataset(self, **kwargs):
        from yolox.data import COCODataset, ValTransform
        legacy = kwargs.get("legacy", False)
        return COCODataset(data_dir=self.data_dir, json_file=getattr(self, "eval_ann", self.val_ann),
                           name=getattr(self, "eval_name", self.val_name),
                           img_size=self.test_size, preproc=ValTransform(legacy=legacy))
'''
open("/content/YOLOX/catface_nano.py", "w").write(exp_src)
!wget -q -nc https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_nano.pth -O /content/YOLOX/yolox_nano.pth
!ls -la /content/YOLOX/yolox_nano.pth


In [ ]:
#@title 7. Train (about 45-70 min on a T4). Progress prints every 10 iterations.
%cd /content/YOLOX
!python tools/train.py -f catface_nano.py -d 1 -b 64 --fp16 -o -c yolox_nano.pth 2>&1 | grep -v "^$" | grep -E "epoch|AP|Average|Error|error|Traceback" | tail -60
!ls -la YOLOX_outputs/catface_nano/


In [ ]:
#@title 8. Evaluate: untouched validation split, then the rotated copy
%cd /content/YOLOX
print("=== validation (as labelled) ===")
!python tools/eval.py -f catface_nano.py -c YOLOX_outputs/catface_nano/best_ckpt.pth -b 64 -d 1 --conf 0.001 --fp16 2>&1 | grep -E "Average Precision|Average Recall" | head -4
# rotated validation: point the same exp at val_rot
rot = open("catface_nano.py").read().replace('self.val_ann = "val.json"', 'self.val_ann = "val_rot.json"').replace('self.val_name = "val"', 'self.val_name = "val_rot"')
open("catface_nano_rot.py", "w").write(rot)
print("=== validation, rotated 20-40 degrees ===")
!python tools/eval.py -f catface_nano_rot.py -c YOLOX_outputs/catface_nano/best_ckpt.pth -b 64 -d 1 --conf 0.001 --fp16 2>&1 | grep -E "Average Precision|Average Recall" | head -4


**How to read this:** the first `Average Precision @[IoU=0.50]` line is the number that matters (a box counts if it overlaps the true face by half). Above ~0.90 on both splits is good. If the rotated split is much lower than the plain one, tell Claude - the rotation share in cell 5 can be raised.

In [ ]:
#@title 9. Export ONNX (decoded boxes inside the graph) and simplify
%cd /content/YOLOX
!python tools/export_onnx.py --output-name catface_yolox_nano.onnx -f catface_nano.py -c YOLOX_outputs/catface_nano/best_ckpt.pth --decode_in_inference 2>&1 | tail -3
!python -m onnxsim catface_yolox_nano.onnx catface_yolox_nano.onnx 2>&1 | tail -3
import os, onnx
m = onnx.load("catface_yolox_nano.onnx")
print("ONNX size: %.1f MB" % (os.path.getsize("catface_yolox_nano.onnx")/1e6))
print("inputs:", [(i.name, [d.dim_value for d in i.type.tensor_type.shape.dim]) for i in m.graph.input])
print("outputs:", [(o.name, [d.dim_value for d in o.type.tensor_type.shape.dim]) for o in m.graph.output])


In [ ]:
#@title 10. Prove the server can run it: cv2.dnn vs PyTorch on validation images
import cv2, numpy as np, json, glob, torch, random
from yolox.utils import postprocess
random.seed(3)
DATA = "/content/YOLOX/datasets/catface"
val = json.load(open(f"{DATA}/annotations/val.json"))
imgs = random.sample(val["images"], 8)

def letterbox(bgr, size=416):
    h, w = bgr.shape[:2]; r = min(size/h, size/w)
    resized = cv2.resize(bgr, (int(w*r), int(h*r)), interpolation=cv2.INTER_LINEAR)
    canvas = np.full((size, size, 3), 114, dtype=np.uint8); canvas[:resized.shape[0], :resized.shape[1]] = resized
    return canvas, r

def decode(out, r, conf=0.3, nms=0.45):
    # out: (N, 6) = cx, cy, w, h, obj, cls  (decode_in_inference=True, 1 class)
    scores = out[:, 4] * out[:, 5]
    keep = scores > conf
    if not keep.any(): return []
    o = out[keep]; s = scores[keep]
    boxes = np.stack([o[:,0]-o[:,2]/2, o[:,1]-o[:,3]/2, o[:,2], o[:,3]], 1) / r
    idx = cv2.dnn.NMSBoxes(boxes.tolist(), s.tolist(), conf, nms)
    return [(tuple(int(v) for v in boxes[i]), float(s[i])) for i in np.array(idx).flatten()]

try:
    net = cv2.dnn.readNetFromONNX("catface_yolox_nano.onnx")
    print("cv2.dnn loaded the model OK")
except Exception as ex:
    net = None; print("cv2.dnn could NOT load the model:", ex, "\n-> the server will need onnxruntime instead; tell Claude.")

from catface_nano import Exp
exp = Exp(); model = exp.get_model().eval()
ckpt = torch.load("YOLOX_outputs/catface_nano/best_ckpt.pth", map_location="cpu"); model.load_state_dict(ckpt["model"])
model.head.decode_in_inference = True

agree = 0; total = 0
for im in imgs:
    bgr = cv2.imread(f"{DATA}/val/{im['file_name']}"); lb, r = letterbox(bgr)
    blob = lb.transpose(2, 0, 1)[None].astype(np.float32)   # BGR, 0-255, no normalisation (YOLOX default)
    with torch.no_grad():
        t_out = model(torch.from_numpy(blob))[0].numpy()
    t_det = decode(t_out, r)
    if net is not None:
        net.setInput(blob); c_out = net.forward()[0]
        c_det = decode(c_out, r)
        same = len(t_det) == len(c_det) and all(abs(a[0][0]-b[0][0]) < 3 for a, b in zip(t_det, c_det))
        agree += same
    total += 1
    gt = [a["bbox"] for a in val["annotations"] if a["image_id"] == im["id"]]
    print(f"{im['file_name'][:28]:28s} torch={[(b, round(s,2)) for b,s in t_det]}  gt={[[int(v) for v in g] for g in gt]}")
if net is not None: print(f"cv2.dnn agrees with torch on {agree}/{total} images")
import time
if net is not None:
    t = time.time()
    for _ in range(10): net.setInput(blob); net.forward()
    print("cv2.dnn CPU-ish latency in Colab: %.0f ms/image" % ((time.time()-t)/10*1000))


In [ ]:
#@title 11. Model card + download the artifacts
%cd /content/YOLOX
import subprocess, datetime, os, json
DATA = "/content/YOLOX/datasets/catface"
ap = subprocess.run("python tools/eval.py -f catface_nano.py -c YOLOX_outputs/catface_nano/best_ckpt.pth -b 64 -d 1 --conf 0.001 --fp16 2>&1 | grep -E 'Average Precision' | head -2", shell=True, capture_output=True, text=True).stdout
ap_rot = subprocess.run("python tools/eval.py -f catface_nano_rot.py -c YOLOX_outputs/catface_nano/best_ckpt.pth -b 64 -d 1 --conf 0.001 --fp16 2>&1 | grep -E 'Average Precision' | head -2", shell=True, capture_output=True, text=True).stdout
card = f'''# catface_yolox_nano.onnx - model card

Trained {datetime.date.today()} with training/catface_detector_colab.ipynb.

- Task: one class, "cat_face", axis-aligned box.
- Architecture: YOLOX-Nano (depth 0.33, width 0.25, depthwise), input 416x416 letterboxed (pad 114), BGR 0-255, no normalisation.
- Output: (1, N, 6) = cx, cy, w, h, objectness, class score, already decoded to input-pixel coordinates.
- Fine-tuned from Megvii's COCO checkpoint yolox_nano.pth (Apache-2.0).
- Data: Roboflow Universe "cat-face-data" (workspace cat-face), CC BY 4.0, {len(json.load(open(DATA+'/annotations/train.json'))['images'])} training images after rotation augmentation (+-15..45 deg on 60% of images).
- Size: {os.path.getsize('catface_yolox_nano.onnx')/1e6:.1f} MB

## Evaluation (COCO AP)
Validation as labelled:
{ap}
Validation rotated 20-40 degrees:
{ap_rot}

## Licenses / attribution
- Dataset: CC BY 4.0 - credit "cat-face-data, Roboflow Universe" (https://universe.roboflow.com/cat-face/cat-face-data)
- YOLOX code and pretrained weights: Apache-2.0 - https://github.com/Megvii-BaseDetection/YOLOX
'''
open("MODEL_CARD.md", "w").write(card); print(card)
from google.colab import files
files.download("catface_yolox_nano.onnx"); files.download("MODEL_CARD.md")


## Done
Put `catface_yolox_nano.onnx` and `MODEL_CARD.md` into `server/models/` in the repo and tell Claude. The server code that loads this model is already in place; it will calibrate the box framing against the landmark model on the test photos before it goes live.